In [ ]:
# %%
# version 3
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import gc

# TODO: We are using linear weights here
# Calculate Sharpe ratio
def calc_spread_return_sharpe(df: pd.DataFrame, portfolio_size: int = 200, toprank_weight_ratio: float = 2) -> float:
    def _calc_spread_return_per_day(df, portfolio_size, toprank_weight_ratio):  # Calculate daily spread return
        assert df['Rank'].min() == 0, "Minimum rank is not 0"
        assert df['Rank'].max() == len(df['Rank']) - 1, "Maximum rank is not equal to the number of stocks minus 1"
        
        # Dynamically adjust portfolio_size
        n_stocks = len(df)
        if n_stocks < portfolio_size:
            portfolio_size = n_stocks  # Adjust portfolio_size if there are not enough stocks
        
        # Select top and bottom portfolio_size stocks by target
        top_stocks = df.sort_values(by='Rank')['Target'][:portfolio_size]
        bottom_stocks = df.sort_values(by='Rank', ascending=False)['Target'][:portfolio_size]
        
        # Generate weights: a linearly decreasing weight array
        weights = np.linspace(start=toprank_weight_ratio, stop=1, num=len(top_stocks))
        
        # Calculate purchase and short positions
        purchase = (top_stocks * weights).sum() / weights.mean()
        short = (bottom_stocks * weights).sum() / weights.mean()
        return purchase - short

    buf = df.groupby('Date').apply(_calc_spread_return_per_day, portfolio_size, toprank_weight_ratio)
    sharpe_ratio = buf.mean() / buf.std()
    return sharpe_ratio


# Read feature data
feature_file = "/kaggle/input/jpx-tokyo-stock-exchange-prediction/train_files/stock_prices.csv"
test_file = "/kaggle/input/jpx-tokyo-stock-exchange-prediction/supplemental_files/stock_prices.csv"
data = pd.read_csv(feature_file)
data2 = pd.read_csv(test_file)
data = data.drop(['ExpectedDividend'], axis=1)
data2 = data2.drop(['ExpectedDividend'], axis=1)

# Drop missing values and sort
data = data.dropna().sort_values(['Date', 'SecuritiesCode'])
data2 = data2.dropna().sort_values(['Date', 'SecuritiesCode'])

# Reset index
data = data.reset_index(drop=True)
data2 = data2.reset_index(drop=True)

# Define feature and target columns
features = [col for col in data.columns if col not in ['Target', 'RowId']]  # Keep 'Date' column
X = data[features]
y = data['Target']

# Reset index for X and y
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

# Split training and test sets
# train_data = data[data['Date'] < '2021-10-01']
# test_data = data[data['Date'] >= '2021-10-01']
train_data = data
test_data = data2

# Extract features and targets for train and test
X_train = train_data[features]
y_train = train_data['Target']
X_test = test_data[features]
y_test = test_data['Target']

print(len(data), len(data2))

# TimeSeries cross-validation
ts_fold = TimeSeriesSplit(n_splits=10, gap=100, test_size=2000)
sharpe_ratio = []
RMSE = []
MAE = []
feat_importance = pd.DataFrame()

for fold, (train_idx, val_idx) in enumerate(ts_fold.split(X_train, y_train)):
    print(f"\n========================== Fold {fold + 1} ==========================")
    
    # Split into training and validation sets
    X_train_fold, y_train_fold = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_valid_fold, y_valid_fold = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Print training and validation date ranges
    print("Train Date range: {} to {}".format(X_train_fold['Date'].min(), X_train_fold['Date'].max()))
    print("Valid Date range: {} to {}".format(X_valid_fold['Date'].min(), X_valid_fold['Date'].max()))
    
    # Drop 'Date' column (not used in training)
    X_train_fold = X_train_fold.drop(['Date'], axis=1)
    X_valid_fold = X_valid_fold.drop(['Date'], axis=1)
    
    # LightGBM parameters
    params = {
        'n_estimators': 500,
        'num_leaves': 100,
        'learning_rate': 0.1,
        'colsample_bytree': 0.9,
        'subsample': 0.8,
        'reg_alpha': 0.4,
        'metric': 'mae',
        'random_state': 21,
        'verbosity': -1
    }
    
    # Train model
    model = LGBMRegressor(**params)
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_train_fold, y_train_fold), (X_valid_fold, y_valid_fold)],
        eval_metric='mae'
    )
    
    # Predict on validation set
    y_pred = model.predict(X_valid_fold)
    rmse = np.sqrt(mean_squared_error(y_valid_fold, y_pred))
    mae = mean_absolute_error(y_valid_fold, y_pred)
    
    # Save feature importances
    feat_importance[f"Importance_Fold{fold + 1}"] = model.feature_importances_
    feat_importance.set_index(X_train_fold.columns, inplace=True)
    
    # Rank predictions per day
    rank = []
    X_valid_df = X_train.iloc[val_idx].copy()
    for date in X_valid_df['Date'].unique():
        temp_df = X_valid_df[X_valid_df['Date'] == date].drop(['Date'], axis=1)
        temp_df['pred'] = model.predict(temp_df)
        temp_df['Rank'] = (temp_df['pred'].rank(method='first', ascending=False) - 1).astype(int)
        rank.append(temp_df['Rank'].values)
    
    # Combine ranks
    stock_rank = pd.Series([x for y in rank for x in y], name='Rank')
    df = pd.concat([X_valid_df.reset_index(drop=True), stock_rank, y_valid_fold.reset_index(drop=True)], axis=1)
    
    # Calculate Sharpe ratio
    sharpe = calc_spread_return_sharpe(df)
    sharpe_ratio.append(sharpe)
    RMSE.append(rmse)
    MAE.append(mae)
    print(f"Valid Sharpe: {sharpe}, RMSE: {rmse}, MAE: {mae}")
    
    # Release memory
    del X_train_fold, y_train_fold, X_valid_fold, y_valid_fold
    gc.collect()

# Print average Sharpe ratio
print(f"\nAverage cross-validation Sharpe Ratio: {np.mean(sharpe_ratio):.4f}, standard deviation = {np.std(sharpe_ratio):.2f}, average RMSE = {np.mean(RMSE)}, , average MAE = {np.mean(MAE)}")

# Evaluate on test set
X_test = X_test.drop(['Date'], axis=1)
y_pred_test = model.predict(X_test)

# Rank and calculate Sharpe on test set
X_test_df = test_data.copy()
X_test_df['pred'] = y_pred_test
X_test_df['Rank'] = X_test_df.groupby('Date')['pred'].rank(method='first', ascending=False) - 1

test_sharpe = calc_spread_return_sharpe(X_test_df)
print(f"Test Sharpe Ratio: {test_sharpe}")

# Save predictions
submission = X_test_df[['Date', 'SecuritiesCode', 'Rank']]
submission.to_csv("submission.csv", index=False)
print("Predictions saved to submission.csv")

# Compute test RMSE and MAE
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Test Sharpe Ratio: {test_sharpe:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE: {test_mae:.4f}")


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013692 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6932
[LightGBM] [Info] Number of data points in the train set: 1774612, number of used features: 31
[LightGBM] [Info] Start training from score 0.000352
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
Test RMSE: 0.0217
Test MAE: 0.0149


C:\Users\zixuanmu\AppData\Local\Temp\1\ipykernel_11144\1930334421.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  buf = df.groupby('Date').apply(_calc_spread_return_per_day, portfolio_size, toprank_weight_ratio)


Test Sharpe Ratio: 0.2308

Submission saved to submission.csv
